# AMATH141 LAB5 — GDP interpolation in monomial form

Build and visualise monomial interpolants for the short and extended GDP datasets, and discuss what the figures tell us about the numerical stability of the monomial basis.

## Question 1 — Plot the monomial interpolant for Table 1

**Question.** Using the dataset from Table 1 of the tutorial,

```
Years: 1980, 1990, 2000, 2010
GDP  : 11.214357, 22.943763, 33.621027, 66.036212
```

build the interpolating polynomial in monomial form and plot it together with the data points. Mark the estimated value at $1992$ on the figure (reference value $\approx 25.465014$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

years = np.array([1980, 1990, 2000, 2010], dtype=float)
gdp = np.array([11.214357, 22.943763, 33.621027, 66.036212], dtype=float)
target_year = 1992.0

# Build the monomial interpolant, evaluate it on a dense grid,
# then plot the data points, the curve, and the estimate at 1992.

# Center the years for numerical stability (Vandermonde is ill-conditioned)
x0 = years[0]
x_shift = years - x0

# Build Vandermonde matrix and solve for monomial coefficients [a0, a1, ..., an]
n = len(years)
V = np.array([[xi**j for j in range(n)] for xi in x_shift], dtype=float)
coeffs = np.linalg.solve(V, gdp)

# Horner evaluation
def horner_eval(coeffs, t):
    result = 0.0
    for a in reversed(coeffs):
        result = result * t + a
    return result

# Dense grid
t_grid = np.linspace(years[0], years[-1], 400)
P_grid = np.array([horner_eval(coeffs, t - x0) for t in t_grid])

# Estimate at 1992
P_star = horner_eval(coeffs, target_year - x0)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_grid, P_grid, 'b-', lw=2, label=r'Monomial interpolant $P_3(x)$')
ax.plot(years, gdp, 'ko', markersize=8, label='Data points (Table 1)')
ax.plot(target_year, P_star, 'rs', markersize=10,
        label=f'Estimate at {int(target_year)}: {P_star:.4f}')
ax.axvline(target_year, color='r', linestyle='--', alpha=0.4)

ax.set_xlabel('Year')
ax.set_ylabel('GDP (trillion USD)')
ax.set_title('Monomial interpolation of GDP — Table 1')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Estimated GDP at {int(target_year)}: {P_star:.6f}")
print(f"Reference value:        25.465014")

## Question 2 — Plot the monomial interpolant for Table 2

**Question.** Using the extended dataset from Table 2,

```
Years: 1960, 1965, 1970, 1975, 1980, 1985, 1990, 1995,
       2000, 2005, 2010, 2015, 2020, 2025
GDP  : 1.366676, 1.995647, 3.409880, 5.916540,
       11.214357, 12.526402, 22.943763, 29.678994,
       33.621027, 47.732965, 66.036212, 75.187709,
       84.906994, 105.000000
```

build the interpolating polynomial in monomial form and plot it together with the data points. Mark the estimated value at $1992$. Do **not** hide oscillations or unusual growth by choosing an overly narrow plotting window.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

years = np.array([1960, 1965, 1970, 1975, 1980, 1985, 1990, 1995,
                  2000, 2005, 2010, 2015, 2020, 2025], dtype=float)
gdp = np.array([1.366676, 1.995647, 3.409880, 5.916540,
                11.214357, 12.526402, 22.943763, 29.678994,
                33.621027, 47.732965, 66.036212, 75.187709,
                84.906994, 105.000000], dtype=float)
target_year = 1992.0

# Build the monomial interpolant, evaluate it on a dense grid,
# then plot the data points, the curve, and the estimate at 1992.

# Center the years for numerical stability — the Vandermonde matrix
# for n=14 nodes spanning 65 years is severely ill-conditioned.
x0 = years.mean()
x_shift = years - x0

# Build Vandermonde matrix V_{i,j} = x_i^j and solve for [a0, ..., an]
n = len(years)
V = np.array([[xi**j for j in range(n)] for xi in x_shift], dtype=float)
coeffs = np.linalg.solve(V, gdp)

# Horner evaluation
def horner_eval(coeffs, t):
    result = 0.0
    for a in reversed(coeffs):
        result = result * t + a
    return result

# Dense grid spanning the full data range — do NOT clip to hide oscillations
t_grid = np.linspace(years[0], years[-1], 2000)
P_grid = np.array([horner_eval(coeffs, t - x0) for t in t_grid])

# Estimate at 1992
P_star = horner_eval(coeffs, target_year - x0)

# Plot
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(t_grid, P_grid, 'b-', lw=2,
        label=f'Monomial interpolant $P_{{{n-1}}}(x)$')
ax.plot(years, gdp, 'ko', markersize=8, label='Data points (Table 2)')
ax.plot(target_year, P_star, 'rs', markersize=10,
        label=f'Estimate at {int(target_year)}: {P_star:.4f}')
ax.axvline(target_year, color='r', linestyle='--', alpha=0.4)

ax.set_xlabel('Year')
ax.set_ylabel('GDP (trillion USD)')
ax.set_title('Monomial interpolation of GDP — Table 2 (extended, degree 13)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Estimated GDP at {int(target_year)}: {P_star:.6f}")
print(f"Reference value:                    25.465014")

## Question 3 — Interpret the monomial plots

**Question.** Compare the two monomial-form figures.
- What changes when you move from Table 1 to Table 2?
- Is the estimate near $1992$ still equally trustworthy?
- What does this suggest about the numerical stability of the monomial form on larger datasets?

### Answer

Moving from Table 1 to Table 2 raises the polynomial degree from $3$ to $13$, and the figures show this clearly: the Table 1 cubic is a smooth curve passing through all four points, while the Table 2 plot exhibits visible Runge-type oscillations between the nodes — especially near the endpoints $1960$–$1965$ and $2015$–$2025$ — where the curve dips and overshoots far from any plausible GDP value. Consequently, the estimate at $1992$ is no longer equally trustworthy: in Table 1 it lies on a smooth segment matching the reference $\approx 25.465014$, but in Table 2 the year $1992$ sits inside an oscillating region of $P_{13}$, so the value read off the curve cannot be relied upon even though we used *more* data. This illustrates that the monomial form does not benefit from additional nodes: each new point increases the degree of $P_n$ and worsens the conditioning of the Vandermonde matrix $V_{i,j} = x_i^{\,j}$, whose condition number grows essentially exponentially with $n$. The practical conclusion is that the monomial form is numerically unstable on larger datasets — for $n$ beyond a handful of points, one should switch to a better-conditioned representation such as Lagrange or Newton form, or use piecewise/spline interpolation.